# MesoNet → TFLite, with an honest accuracy story

Produces `meso4_df.tflite` for Reality Lock's optional "AI confidence score"
layer (Phase 7, `research/04` §2/§6), plus the numbers needed to describe it
truthfully.

## Read this before running

This notebook does **two separate things**, and they must not be confused:

1. **Conversion + parity** (sections 1–5). Fully automatic. Proves the TFLite
   file computes the same function as the original Keras model. This is a
   statement about *the conversion*, not about deepfake detection.
2. **Accuracy measurement** (section 6). Requires **you** to supply a labelled
   dataset. Without it there is no accuracy figure, and none may be invented.

The MesoNet repository ships **4 test images** (2 fake, 2 real). Four images
cannot support any accuracy claim — a coin flip gets 4/4 about 6% of the time.
They are used here only to confirm the model loaded and the label convention is
right.

### What you may claim if you only run sections 1–5

> An experimental Meso-4 classifier (Afchar et al., IEEE WIFS 2018) is bundled,
> using the authors' published Deepfake-set weights. The conversion to TFLite
> was verified to preserve outputs to within *N* (section 5). **No accuracy has
> been measured on this project's data; the published figures are the authors'
> and were obtained on their own dataset.**

That is a real, defensible claim. "~90% accurate" is not, unless section 6 was
run on a labelled set and *that* number is quoted with its dataset named.

### Model facts (from the authors' `classifiers.py` and `example.py`)

| | |
|---|---|
| Input | 256×256×3, float, rescaled to `[0, 1]` (i.e. `/255`) |
| Output | 1 unit, sigmoid |
| **Convention** | **→1 = REAL, →0 = DEEPFAKE** |
| Parameters | ~28k (Meso-4) |

The convention comes from `flow_from_directory` sorting class folders
alphabetically — `df` → 0, `real` → 1. Getting it backwards would invert every
prediction while still looking plausible, so section 4 checks it against the
authors' own images rather than trusting this table.

## 1. Environment

`Meso4_DF.h5` is a **legacy Keras 2 weights file**. Modern TensorFlow ships
Keras 3, whose `.h5` handling differs and whose models need an extra export step
before `TFLiteConverter` will take them.

`TF_USE_LEGACY_KERAS=1` + the `tf_keras` package is TensorFlow's own documented
escape hatch: it makes `tf.keras` resolve to Keras 2, so both the weight load and
`from_keras_model` work on the path they were written for. **The env var must be
set before `import tensorflow`** — that is why it is in its own cell.

In [ ]:
!pip install -q tf-keras

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import numpy as np

print("tensorflow", tf.__version__)
print("tf.keras  ", tf.keras.__version__)
assert tf.keras.__version__.startswith("2."), (
    "tf.keras is not Keras 2. Restart the runtime and run this cell FIRST — "
    "TF_USE_LEGACY_KERAS is only read at import time."
)

## 2. Weights

Only the weights file is downloaded, not the repository. `example.py` imports
`pipeline.py`, which pulls in `face_recognition`/`dlib` for video handling — a
long, fragile build that this notebook does not need. The architecture is
reproduced in section 3 instead, and section 3 asserts that it matches.

In [ ]:
BASE = "https://github.com/DariusAf/MesoNet/raw/master"

# Meso4_DF = trained on the Deepfake set. Meso4_F2F = Face2Face.
# DF is the right default: it is the manipulation family the PPT's threat model
# describes. Swap the name to build the other one.
WEIGHTS = "Meso4_DF.h5"

!wget -q -O {WEIGHTS} {BASE}/weights/{WEIGHTS}

# The authors' four sample images, used ONLY to check the model loaded and the
# label convention is right. Not an evaluation set — see section 6.
!mkdir -p test_images/df test_images/real
!wget -q -O test_images/df/df00204.jpg     {BASE}/test_images/df/df00204.jpg
!wget -q -O test_images/df/df01254.jpg     {BASE}/test_images/df/df01254.jpg
!wget -q -O test_images/real/real00240.jpg {BASE}/test_images/real/real00240.jpg
!wget -q -O test_images/real/real00772.jpg {BASE}/test_images/real/real00772.jpg

!ls -la {WEIGHTS} && du -sh test_images

## 3. The Meso-4 architecture

Transcribed from `DariusAf/MesoNet/classifiers.py`. `load_weights` on a legacy
`.h5` matches **by layer order and shape**, so a transcription error would not
necessarily throw — it could load into the wrong layer and quietly produce a
model that predicts nonsense. The parameter-count assertion below is the cheap
guard against that; section 4 is the real one.

In [ ]:
from tensorflow.keras.models import Model as KerasModel
from tensorflow.keras.layers import (
    Input, Dense, Flatten, Conv2D, MaxPooling2D, BatchNormalization, Dropout, LeakyReLU,
)

IMGWIDTH = 256

def build_meso4():
    x = Input(shape=(IMGWIDTH, IMGWIDTH, 3))

    x1 = Conv2D(8, (3, 3), padding="same", activation="relu")(x)
    x1 = BatchNormalization()(x1)
    x1 = MaxPooling2D(pool_size=(2, 2), padding="same")(x1)

    x2 = Conv2D(8, (5, 5), padding="same", activation="relu")(x1)
    x2 = BatchNormalization()(x2)
    x2 = MaxPooling2D(pool_size=(2, 2), padding="same")(x2)

    x3 = Conv2D(16, (5, 5), padding="same", activation="relu")(x2)
    x3 = BatchNormalization()(x3)
    x3 = MaxPooling2D(pool_size=(2, 2), padding="same")(x3)

    x4 = Conv2D(16, (5, 5), padding="same", activation="relu")(x3)
    x4 = BatchNormalization()(x4)
    x4 = MaxPooling2D(pool_size=(4, 4), padding="same")(x4)

    y = Flatten()(x4)
    y = Dropout(0.5)(y)
    y = Dense(16)(y)
    y = LeakyReLU(alpha=0.1)(y)
    y = Dropout(0.5)(y)
    y = Dense(1, activation="sigmoid")(y)
    return KerasModel(inputs=x, outputs=y)

model = build_meso4()
model.load_weights(WEIGHTS)

total = model.count_params()
print("parameters:", total)
# The paper's 27,977 is the TRAINABLE count. Keras reports 28,073 total, and
# the 96 difference is exactly the BatchNorm moving_mean/moving_var pairs
# (2x8 + 2x8 + 2x16 + 2x16), which are state rather than learned weights. Both
# are asserted, because matching total-but-not-trainable would mean the layer
# shapes are right while the batch-norm layers are wrong.
trainable = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
print("trainable :", trainable, "(paper: 27,977)")
assert total == 28_073, f"expected 28,073 total parameters, got {total}"
assert trainable == 27_977, f"expected 27,977 trainable parameters, got {trainable}"
model.summary()

## 4. Does it actually work? (label convention check)

The one check that catches a wrongly-transcribed architecture, a bad weight load,
**and** an inverted label convention. If the two `real/` images do not score
higher than the two `df/` images, something upstream is wrong and nothing below
this point is meaningful.

This is a smoke test, not an accuracy measurement.

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import glob

def load_batch(paths):
    """256x256, scaled to [0,1] — exactly the authors' ImageDataGenerator(rescale=1./255).

    `load_img` resamples with **nearest** by default, and that detail is not
    cosmetic. Re-running these four images through the exported .tflite with
    other filters moves the scores by more than the quantisation error the int8
    model was rejected for:

        nearest   max delta vs this notebook = 0.0000   <- what the model expects
        bicubic                              = 0.0134
        bilinear                             = 0.0284

    Bilinear is the default in most Android image pipelines, so an integration
    that resizes "the obvious way" silently evaluates a different image than the
    one measured here. Whatever consumes this model must pin its resampling as
    deliberately as it pins the model file.
    """
    arrays = [img_to_array(load_img(p, target_size=(IMGWIDTH, IMGWIDTH))) / 255.0 for p in paths]
    return np.stack(arrays).astype(np.float32)

fake_paths = sorted(glob.glob("test_images/df/*.jpg"))
real_paths = sorted(glob.glob("test_images/real/*.jpg"))

fake_scores = model.predict(load_batch(fake_paths), verbose=0).ravel()
real_scores = model.predict(load_batch(real_paths), verbose=0).ravel()

for p, s in zip(fake_paths, fake_scores):
    print(f"  DEEPFAKE {os.path.basename(p):<16} score={s:.4f}")
for p, s in zip(real_paths, real_scores):
    print(f"  REAL     {os.path.basename(p):<16} score={s:.4f}")

assert real_scores.min() > fake_scores.max(), (
    "The real images did not score above the fake ones. Either the architecture "
    "transcription is wrong, the weights failed to load, or the convention is "
    "inverted. Do not proceed."
)
print("\nOK — convention confirmed: higher score = more REAL.")

## 5. Convert to TFLite, and prove the conversion is faithful

Two artifacts:

- **float32** — the reference. Use this one unless size is a real problem.
- **dynamic-range int8** — ~4× smaller. Weights are quantised, activations stay
  float. Its parity error is reported separately and *will* be larger.

The parity number matters more than either file. "It converted" is not evidence
the converted model computes the same function, and a silently-degraded model
would be indistinguishable from a working one without this measurement.

In [ ]:
def to_tflite(keras_model, quantize=False):
    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    if quantize:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    return converter.convert()

float_model = to_tflite(model, quantize=False)
int8_model = to_tflite(model, quantize=True)

open("meso4_df.tflite", "wb").write(float_model)
open("meso4_df_int8.tflite", "wb").write(int8_model)

print(f"float32 : {len(float_model)/1024:8.1f} KiB  -> meso4_df.tflite")
print(f"int8    : {len(int8_model)/1024:8.1f} KiB  -> meso4_df_int8.tflite")

In [ ]:
def tflite_predict(model_bytes, batch):
    interpreter = tf.lite.Interpreter(model_content=model_bytes)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    results = []
    for row in batch:  # default TFLite signature is batch size 1
        interpreter.set_tensor(inp["index"], row[None, ...])
        interpreter.invoke()
        results.append(interpreter.get_tensor(out["index"]).ravel()[0])
    return np.array(results)

# Random inputs, not the 4 sample images: parity must hold across the input
# space, and four faces exercise a vanishingly small part of it.
rng = np.random.default_rng(0)
probe = rng.random((32, IMGWIDTH, IMGWIDTH, 3), dtype=np.float32)

keras_out = model.predict(probe, verbose=0).ravel()
float_err = np.abs(keras_out - tflite_predict(float_model, probe)).max()
int8_err = np.abs(keras_out - tflite_predict(int8_model, probe)).max()

print(f"max |keras - tflite_float32| = {float_err:.3e}")
print(f"max |keras - tflite_int8|    = {int8_err:.3e}")

assert float_err < 1e-4, f"float32 conversion is not faithful ({float_err:.3e})"
print("\nQuote the float32 figure as the conversion-fidelity number.")
if int8_err > 0.05:
    print(
        f"NOTE: int8 differs by up to {int8_err:.3f} on a 0-1 score. That is large "
        "enough to flip a borderline call — prefer the float32 model, which is "
        "only ~110 KiB anyway."
    )

## 6. Accuracy — the part that needs your data

**Everything above is about the conversion. This section is the only one that
produces an accuracy figure, and it cannot run without a labelled dataset.**

Arrange images as:

```
eval_data/
  df/     ← known manipulated
  real/   ← known authentic
```

Then set `EVAL_DIR` below and run. Options:

- **FaceForensics++** — what Meso-4 was trained against; needs a signed academic
  request form.
- **Kaggle `xhlulu/140k-real-and-fake-faces`** — instant with a Kaggle token, but
  it is StyleGAN faces, **not** the face-swap manipulations Meso4_DF was trained
  on. Expect worse numbers, and say which dataset you used when quoting them.
- **DFDC sample set** — closest to the training distribution, ~4 GB.

Two things to report honestly whichever you use:

1. **The dataset name, alongside every number.** Accuracy without a named dataset
   is not a measurement.
2. **The operating threshold.** 0.5 is a default, not a law; the sweep below
   shows what you trade at each one. For Reality Lock the expensive error is a
   **false "fake"** on a genuine photo, so a threshold with high specificity is
   worth more than one with high raw accuracy.

In [ ]:
EVAL_DIR = None  # e.g. "eval_data"  <-- set this to measure accuracy

if EVAL_DIR is None:
    print(
        "No dataset supplied, so NO accuracy was measured.\n\n"
        "Do not quote an accuracy figure for this model. Quote the conversion\n"
        "fidelity from section 5 and cite the authors' published results as\n"
        "theirs, on their data."
    )
else:
    fake_files = sorted(glob.glob(f"{EVAL_DIR}/df/*"))
    real_files = sorted(glob.glob(f"{EVAL_DIR}/real/*"))
    print(f"{len(fake_files)} fake, {len(real_files)} real")
    assert fake_files and real_files, "both df/ and real/ must contain images"

    def scores_for(paths, chunk=64):
        out = []
        for i in range(0, len(paths), chunk):
            out.append(model.predict(load_batch(paths[i:i + chunk]), verbose=0).ravel())
        return np.concatenate(out)

    s_fake, s_real = scores_for(fake_files), scores_for(real_files)
    y_true = np.concatenate([np.zeros(len(s_fake)), np.ones(len(s_real))])
    y_score = np.concatenate([s_fake, s_real])

    print(f"\n{'thresh':>7} {'acc':>7} {'sens':>7} {'spec':>7}   (sens=caught fakes, spec=real kept)")
    for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        pred = (y_score >= t).astype(int)
        acc = (pred == y_true).mean()
        sens = (pred[y_true == 0] == 0).mean()   # fakes correctly called fake
        spec = (pred[y_true == 1] == 1).mean()   # real correctly kept real
        print(f"{t:7.2f} {acc:7.3f} {sens:7.3f} {spec:7.3f}")

    try:
        from sklearn.metrics import roc_auc_score
        # Threshold-independent, so it is the fairest single number to quote.
        print(f"\nROC AUC = {roc_auc_score(y_true, y_score):.4f}")
    except ImportError:
        pass

    print(
        f"\nReport as: 'Meso-4 (Meso4_DF weights), evaluated on <DATASET NAME>, "
        f"n={len(y_true)}' — with the threshold you chose and why."
    )

## 7. Download

Put `meso4_df.tflite` in `android/app/src/main/assets/`.

One integration rule, from `research/04` §6 and ADR-0005: this is a **secondary,
clearly-experimental** signal. It never replaces the ELA/EXIF layer, and it never
contributes to a proof package's verdict — a proof package's verdict is about
whether bytes were altered since capture, which a classifier has no opinion on.
It also must never be labelled with an accuracy this project did not measure.

In [ ]:
from google.colab import files

files.download("meso4_df.tflite")
# files.download("meso4_df_int8.tflite")  # only if size genuinely matters